In [2]:
!pip install biopython


In [4]:
import os
import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser

# 🔧 CHANGE this to your actual folder path with the CIF files
STRUCT_DIR = r"D:\BBB Trans AI\af3_model_output"

# Where to save the structural feature table
OUT_CSV = r"D:\BBB Trans AI\features\bbb_struct_features_100.csv"

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# AA sets for simple composition features
POSITIVE = {"LYS", "ARG", "HIS"}
NEGATIVE = {"ASP", "GLU"}
HYDROPHOBIC = {"ALA", "VAL", "ILE", "LEU", "MET", "PHE", "TRP", "TYR", "PRO"}
AROMATIC = {"PHE", "TYR", "TRP"}

parser = MMCIFParser(QUIET=True)
rows = []

for fname in os.listdir(STRUCT_DIR):
    if not fname.lower().endswith(".cif"):
        continue

    # Example filename: n_c2_model_0.cif  ->  ID = N_C2
    base = fname.replace("_model_0.cif", "")
    pep_id = base.upper()

    path = os.path.join(STRUCT_DIR, fname)
    print("Processing", fname, "-> ID:", pep_id)

    try:
        structure = parser.get_structure(pep_id, path)
    except Exception as e:
        print("  ❌ Failed to parse:", e)
        continue

    model = structure[0]

    # ---- Atom-based: radius of gyration ----
    coords = np.array([
        atom.coord
        for atom in model.get_atoms()
        if atom.element != "H"
    ])

    if coords.size == 0:
        print("  ⚠ No atoms, skipping.")
        continue

    center = coords.mean(axis=0)
    rg = np.sqrt(np.mean(np.sum((coords - center) ** 2, axis=1)))

    # ---- Residue-based composition ----
    residues = [res for res in model.get_residues() if res.id[0] == " "]
    L = len(residues)

    pos = neg = hyd = arom = 0
    for res in residues:
        resname = res.get_resname().strip().upper()
        if resname in POSITIVE: pos += 1
        if resname in NEGATIVE: neg += 1
        if resname in HYDROPHOBIC: hyd += 1
        if resname in AROMATIC: arom += 1

    frac_pos = pos / L if L > 0 else np.nan
    frac_neg = neg / L if L > 0 else np.nan
    frac_hyd = hyd / L if L > 0 else np.nan
    frac_arom = arom / L if L > 0 else np.nan
    net_charge = pos - neg
    compactness = L / rg if rg > 0 else np.nan

    rows.append({
        "ID": pep_id,
        "length": L,
        "radius_gyration": rg,
        "compactness": compactness,
        "frac_positive": frac_pos,
        "frac_negative": frac_neg,
        "net_charge": net_charge,
        "frac_hydrophobic": frac_hyd,
        "frac_aromatic": frac_arom,
    })

df_struct = pd.DataFrame(rows).sort_values("ID").reset_index(drop=True)
df_struct.to_csv(OUT_CSV, index=False)

print("\n✅ Saved structural features to:", OUT_CSV)
print("Shape:", df_struct.shape)
df_struct.head()


Processing n_c108_model_0.cif -> ID: N_C108
Processing n_c112_model_0.cif -> ID: N_C112
Processing n_c115_model_0.cif -> ID: N_C115
Processing n_c11_model_0.cif -> ID: N_C11
Processing n_c123_model_0.cif -> ID: N_C123
Processing n_c131_model_0.cif -> ID: N_C131
Processing n_c135_model_0.cif -> ID: N_C135
Processing n_c13_model_0.cif -> ID: N_C13
Processing n_c14_model_0.cif -> ID: N_C14
Processing n_c167_model_0.cif -> ID: N_C167
Processing n_c168_model_0.cif -> ID: N_C168
Processing n_c169_model_0.cif -> ID: N_C169
Processing n_c170_model_0.cif -> ID: N_C170
Processing n_c187_model_0.cif -> ID: N_C187
Processing n_c18_model_0.cif -> ID: N_C18
Processing n_c192_model_0.cif -> ID: N_C192
Processing n_c193_model_0.cif -> ID: N_C193
Processing n_c197_model_0.cif -> ID: N_C197
Processing n_c198_model_0.cif -> ID: N_C198
Processing n_c20_model_0.cif -> ID: N_C20
Processing n_c210_model_0.cif -> ID: N_C210
Processing n_c211_model_0.cif -> ID: N_C211
Processing n_c216_model_0.cif -> ID: N_C21

,ID,length,radius_gyration,compactness,frac_positive,frac_negative,net_charge,frac_hydrophobic,frac_aromatic
0,N_C108,12,8.590800,1.396843,0.750000,0.000000,9,0.166667,0.00
1,N_C11,20,9.416270,2.123983,0.150000,0.000000,3,0.650000,0.05
2,N_C112,24,11.112412,2.159747,0.416667,0.041667,9,0.250000,0.00
3,N_C115,8,6.535278,1.224125,0.500000,0.000000,4,0.500000,0.50
4,N_C123,15,10.281929,1.458870,0.266667,0.000000,4,0.466667,0.20
